# Election Bloc Change Prediction Project
## Notebook 01 — Historical election loading and party-to-bloc mapping

Loads Knesset elections 16–25, aggregates ballot-level files to locality level, reconstructs Knesset 17 locality symbols from names, excludes special-envelope rows, and audits every party mapping.

The mapping is election-specific because ballot letters and coalitional meaning change over time. Minor unmapped lists remain under `Other`; any unmapped list above 2% stops the notebook.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 300)
REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO = Path("/content/Election_Bloc_Prediction_Project")

def locate_repo():
    candidates=[]
    if os.getenv("ELECTION_PROJECT_ROOT"):
        candidates.append(Path(os.environ["ELECTION_PROJECT_ROOT"]).expanduser())
    cwd=Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, DEFAULT_REPO]
    for p in candidates:
        if (p/"data"/"raw").exists(): return p
    if Path("/content").exists():
        if DEFAULT_REPO.exists(): shutil.rmtree(DEFAULT_REPO)
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(DEFAULT_REPO)],check=True)
        return DEFAULT_REPO
    raise FileNotFoundError("Repository not found. Set ELECTION_PROJECT_ROOT.")

REPO_ROOT=locate_repo()
RAW_DIR=REPO_ROOT/"data"/"raw"
INTERIM_DIR=REPO_ROOT/"data"/"interim"
PROCESSED_DIR=REPO_ROOT/"data"/"processed"
REPORTS_DIR=REPO_ROOT/"reports"
TABLES_DIR=REPORTS_DIR/"tables"
FIGURES_DIR=REPORTS_DIR/"figures"
SUMMARIES_DIR=REPORTS_DIR/"summaries"
MODELS_DIR=REPO_ROOT/"models"
for d in [INTERIM_DIR,PROCESSED_DIR,TABLES_DIR,FIGURES_DIR,SUMMARIES_DIR,MODELS_DIR]: d.mkdir(parents=True,exist_ok=True)
print("Repository:",REPO_ROOT)

## 1. Election inventory and metadata

In [ ]:
ELECTION_DIR=RAW_DIR/"Election_data"
ELECTION_FILES={
 "Knesset_16":"Knesset_16_Election_Results_2003_By_Kalfi.csv.xls",
 "Knesset_17":"Knesset_17_Election_Results_2006_By_Kalfi.csv.xls",
 "Knesset_18":"Knesset_18_Election_Results_2009_By_Kalfi.csv.csv",
 "Knesset_19":"Knesset_19_Election_Results_2013_By_Locality.csv.csv",
 "Knesset_20":"Knesset_20_Election_Results_2015_By_Locality.csv.csv",
 "Knesset_21":"Knesset_21_Election_Results_2019_By_Locality.csv",
 "Knesset_22":"Knesset_22_Election_Results_2019_By_Locality.csv",
 "Knesset_23":"Knesset_23_Election_Results_2020_By_Locality.csv",
 "Knesset_24":"Knesset_24_Election_Results_2021_By_Locality.csv",
 "Knesset_25":"Knesset_25_Election_Results_2022_By_Locality.csv",
}
ELECTION_YEARS={16:2003,17:2006,18:2009,19:2013,20:2015,21:2019,22:2019,23:2020,24:2021,25:2022}
ELECTION_METADATA={f"Knesset_{n}":{"election_number":n,"year":y} for n,y in ELECTION_YEARS.items()}
missing=[str(ELECTION_DIR/f) for f in ELECTION_FILES.values() if not (ELECTION_DIR/f).exists()]
if missing and (REPO_ROOT/".git").exists():
    subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin","main"],check=True)
    missing=[str(ELECTION_DIR/f) for f in ELECTION_FILES.values() if not (ELECTION_DIR/f).exists()]
if missing: raise FileNotFoundError("Missing election files:\n"+"\n".join(missing))
print("All",len(ELECTION_FILES),"election files found.")

## 2. Robust readers and locality crosswalk

In [ ]:
def clean_col(x): return re.sub(r"\s+"," ",str(x).replace("\ufeff","").strip())
def num(s):
    return pd.to_numeric(s.astype("string").str.replace(",","",regex=False).str.strip(),errors="coerce")
def read_any(path):
    suffix=path.suffix.lower()
    if suffix in {".xls",".xlsx"} or path.name.endswith(".csv.xls"):
        df=pd.read_excel(path)
        return df,"excel"
    errors=[]
    for enc in ["utf-8-sig","utf-8","cp1255","iso-8859-8"]:
        try: return pd.read_csv(path,encoding=enc),enc
        except UnicodeDecodeError as e: errors.append(str(e))
    raise UnicodeError(f"Could not decode {path}: {errors[-1]}")
def first_col(df,candidates,required=True):
    for c in candidates:
        if c in df.columns:return c
    if required: raise KeyError(f"Missing one of {candidates}")
    return None
def norm_name(x):
    if pd.isna(x): return pd.NA
    x=unicodedata.normalize("NFKD",str(x)).replace("׳","'").replace("״",'"')
    x=re.sub(r"[\(\)\[\]\{\}'\".,]"," ",x)
    x=x.replace("־","-")
    x=re.sub(r"\s*-\s*","-",x)
    x=re.sub(r"\s+"," ",x).strip()
    return x

raw={}; inventory=[]
for election,filename in ELECTION_FILES.items():
    df,reader=read_any(ELECTION_DIR/filename)
    df.columns=[clean_col(c) for c in df.columns]
    raw[election]=df
    inventory.append({"election":election,"file":filename,"reader":reader,"rows":len(df),"columns":df.shape[1]})
pd.DataFrame(inventory)

In [ ]:
# Build a symbol reference from every election that already contains locality symbols.
ref=[]
for election,df in raw.items():
    sc=first_col(df,["סמל ישוב","סמל יישוב","סמל היישוב"],required=False)
    nc=first_col(df,["שם ישוב","שם יישוב","שם היישוב"],required=False)
    if sc and nc:
        tmp=pd.DataFrame({"locality_symbol":num(df[sc]).astype("Int64"),"locality_name":df[nc].astype("string")})
        tmp=tmp.loc[tmp.locality_symbol.gt(0)&tmp.locality_symbol.ne(9999)].dropna().drop_duplicates()
        tmp["normalized_name"]=tmp.locality_name.map(norm_name)
        ref.append(tmp)
reference=pd.concat(ref,ignore_index=True).drop_duplicates()
name_candidates=(reference.groupby("normalized_name")["locality_symbol"].agg(lambda s: sorted(set(int(v) for v in s.dropna()))).reset_index())
unique_name_map={r.normalized_name:r.locality_symbol[0] for r in name_candidates.itertuples() if len(r.locality_symbol)==1}
MANUAL_NAME_ALIASES={
    norm_name("יהוד-נווה אפרים"):9400,
    norm_name("סוואעד חמיירה"):942,
    norm_name("צורן-קדימה"):195,
}
print("Unique normalized names in reference:",len(unique_name_map))

## 3. Election-specific party mappings

In [ ]:
MODELED_BLOCS=["Right","Center_Left","Haredi","Arab"]
YISRAEL_BEITEINU_BLOC="Center_Left"
rows=[]
def add(e,bloc,items):
    for letter,name in items: rows.append((f"Knesset_{e}",letter,name,bloc))
# 16
add(16,"Right",[("מחל","Likud"),("ב","National Religious Party"),("ל","National Union"),("כן","Yisrael BaAliyah"),("נץ","Herut")])
add(16,"Center_Left",[("אמת","Labor-Meimad"),("יש","Shinui"),("מרצ","Meretz"),("ם","One Nation")])
add(16,"Haredi",[("שס","Shas"),("ג","United Torah Judaism")])
add(16,"Arab",[("ו","Hadash"),("ד","Balad"),("עם","United Arab List")])
# 17
add(17,"Right",[("מחל","Likud"),("טב","National Union-NRP"),("כ","Jewish National Front"),("נץ","Herut")])
add(17,"Center_Left",[("כן","Kadima"),("אמת","Labor-Meimad"),("מרצ","Meretz-Yachad"),("זך","Gil Pensioners"),("חץ","Hetz"),("יש","Shinui"),("ל","Yisrael Beiteinu")])
add(17,"Haredi",[("שס","Shas"),("ג","United Torah Judaism")])
add(17,"Arab",[("עם","United Arab List-Ta'al"),("ו","Hadash"),("ד","Balad"),("קפ","Arab National Party"),("ק","Da'am")])
# 18
add(18,"Right",[("מחל","Likud"),("ט","National Union"),("ב","Jewish Home"),("חי","Ahi")])
add(18,"Center_Left",[("כן","Kadima"),("אמת","Labor"),("מרצ","Meretz"),("ל","Yisrael Beiteinu"),("זך","Gil")])
add(18,"Haredi",[("שס","Shas"),("ג","United Torah Judaism")])
add(18,"Arab",[("עם","United Arab List-Ta'al"),("ו","Hadash"),("ד","Balad"),("ק","Da'am")])
# 19
add(19,"Right",[("מחל","Likud-Yisrael Beiteinu"),("טב","Jewish Home"),("נץ","Otzma LeYisrael")])
add(19,"Center_Left",[("פה","Yesh Atid"),("אמת","Labor"),("צפ","Hatnua"),("מרץ","Meretz"),("כן","Kadima")])
add(19,"Haredi",[("שס","Shas"),("ג","United Torah Judaism"),("ץ","Am Shalem"),("פז","Koach Lehashpia")])
add(19,"Arab",[("עם","United Arab List-Ta'al"),("ו","Hadash"),("ד","Balad"),("ק","Da'am")])
# 20
add(20,"Right",[("מחל","Likud"),("כ","Kulanu"),("טב","Jewish Home"),("קץ","Yachad")])
add(20,"Center_Left",[("אמת","Zionist Union"),("פה","Yesh Atid"),("מרצ","Meretz"),("ל","Yisrael Beiteinu")])
add(20,"Haredi",[("שס","Shas"),("ג","United Torah Judaism")])
add(20,"Arab",[("ודעם","Joint List")])
# 21-25 retain the documented mapping used in the original redesign.
add(21,"Right",[("מחל","Likud"),("טב","Union of Right-Wing Parties"),("כ","Kulanu"),("נ","New Right"),("ז","Zehut")])
add(21,"Center_Left",[("פה","Blue and White"),("אמת","Labor"),("מרצ","Meretz"),("נר","Gesher"),("ל","Yisrael Beiteinu")])
add(21,"Haredi",[("שס","Shas"),("ג","United Torah Judaism")]); add(21,"Arab",[("ום","Hadash-Ta'al"),("דעם","Ra'am-Balad")])
add(22,"Right",[("מחל","Likud"),("טב","Yamina"),("כף","Otzma Yehudit")]); add(22,"Center_Left",[("פה","Blue and White"),("אמת","Labor-Gesher"),("מרצ","Democratic Union"),("ל","Yisrael Beiteinu")]); add(22,"Haredi",[("שס","Shas"),("ג","UTJ")]); add(22,"Arab",[("ודעם","Joint List")])
add(23,"Right",[("מחל","Likud"),("טב","Yamina"),("נץ","Otzma Yehudit")]); add(23,"Center_Left",[("פה","Blue and White"),("אמת","Labor-Gesher-Meretz"),("ל","Yisrael Beiteinu")]); add(23,"Haredi",[("שס","Shas"),("ג","UTJ")]); add(23,"Arab",[("ודעם","Joint List")])
add(24,"Right",[("מחל","Likud"),("ב","Yamina"),("ט","Religious Zionism"),("ת","New Hope")]); add(24,"Center_Left",[("פה","Yesh Atid"),("אמת","Labor"),("מרצ","Meretz"),("כן","Blue and White"),("ל","Yisrael Beiteinu")]); add(24,"Haredi",[("שס","Shas"),("ג","UTJ")]); add(24,"Arab",[("ודעם","Joint List"),("עם","Ra'am")])
add(25,"Right",[("מחל","Likud"),("ב","Jewish Home"),("ט","Religious Zionism-Otzma")]); add(25,"Center_Left",[("פה","Yesh Atid"),("אמת","Labor"),("מרצ","Meretz"),("כן","National Unity"),("ל","Yisrael Beiteinu")]); add(25,"Haredi",[("שס","Shas"),("ג","UTJ")]); add(25,"Arab",[("עם","Ra'am"),("ום","Hadash-Ta'al"),("ד","Balad")])
party_mapping=pd.DataFrame(rows,columns=["target_election","ballot_letter","party_name","bloc"])
if party_mapping.duplicated(["target_election","ballot_letter"]).any(): raise ValueError("Duplicate election-letter mappings")
party_mapping.head()

## 4. Aggregate every election to locality level

In [ ]:
NON_PARTY={"סמל ועדה","שם ישוב","שם יישוב","שם היישוב","סמל ישוב","סמל יישוב","סמל היישוב","סמל קלפי","מספר קלפי","כתובת","בוחרים","בזב","בז''ב","מצביעים","פסולים","כשרים","ריכוז","שופט","ברזל","ת. עדכון"}
def party_cols(df):
    out=[]
    for c in df.columns:
        if c in NON_PARTY or c.lower().startswith("unnamed"): continue
        if num(df[c]).notna().any(): out.append(c)
    return out

def attach_symbol(df,election):
    sc=first_col(df,["סמל ישוב","סמל יישוב","סמל היישוב"],required=False)
    nc=first_col(df,["שם ישוב","שם יישוב","שם היישוב"],required=False)
    names=df[nc].astype("string").str.strip() if nc else pd.Series(pd.NA,index=df.index,dtype="string")
    if sc:
        symbols=num(df[sc]).astype("Int64")
        method=pd.Series("source_symbol",index=df.index,dtype="string")
    else:
        normalized=names.map(norm_name)
        symbols=normalized.map(unique_name_map).fillna(normalized.map(MANUAL_NAME_ALIASES)).astype("Int64")
        method=np.where(normalized.isin(MANUAL_NAME_ALIASES),"manual_alias","normalized_name")
        method=pd.Series(method,index=df.index,dtype="string")
    return symbols,names,method

def aggregate_election(df,election):
    number=ELECTION_METADATA[election]["election_number"]
    symbols,names,match_method=attach_symbol(df,election)
    valid=num(df[first_col(df,["כשרים"])]).fillna(0)
    cast_col=first_col(df,["מצביעים"],required=False); cast=num(df[cast_col]) if cast_col else valid.copy()
    invalid_col=first_col(df,["פסולים"],required=False); invalid=num(df[invalid_col]) if invalid_col else (cast-valid)
    elig_col=first_col(df,["בזב","בז''ב","בוחרים"],required=False); eligible=num(df[elig_col]) if elig_col else pd.Series(np.nan,index=df.index)
    lookup=(party_mapping.loc[party_mapping.target_election.eq(election)].set_index("ballot_letter")[["bloc","party_name"]].to_dict("index"))
    counts=pd.DataFrame(0.0,index=df.index,columns=MODELED_BLOCS)
    for letter,info in lookup.items():
        if letter not in df.columns: raise KeyError(f"{election}: mapped letter {letter} absent")
        counts[info["bloc"]]+=num(df[letter]).fillna(0)
    modeled=counts.sum(axis=1); other=(valid-modeled).clip(lower=0)
    special_name=names.fillna("").str.contains("מעטפות|חיצוניות|כפולות",regex=True)
    keep=symbols.notna()&symbols.gt(0)&symbols.ne(9999)&~special_name
    base=pd.DataFrame({"locality_symbol":symbols,"election_locality_name":names,"symbol_match_method":match_method,"target_election":election,"election_number":number,"year":ELECTION_METADATA[election]["year"],"eligible_voters":eligible,"votes_cast":cast,"invalid_votes":invalid,"valid_votes":valid,"Other":other})
    for b in MODELED_BLOCS: base[b]=counts[b]
    base=base.loc[keep].copy()
    # Aggregate ballots; sum eligible voters for ballot-level files and retain the locality-level value otherwise.
    is_ballot=first_col(df,["סמל קלפי","מספר קלפי"],required=False) is not None
    agg={"election_locality_name":"first","symbol_match_method":"first","target_election":"first","election_number":"first","year":"first","eligible_voters":"sum" if is_ballot else "first","votes_cast":"sum","invalid_votes":"sum","valid_votes":"sum","Other":"sum",**{b:"sum" for b in MODELED_BLOCS}}
    out=base.groupby("locality_symbol",as_index=False).agg(agg)
    out["total_main_blocs"]=out[MODELED_BLOCS].sum(axis=1)
    out["turnout_pct"]=np.where(out.eligible_voters.gt(0),out.votes_cast/out.eligible_voters*100,np.nan)
    out["modeled_vote_coverage_pct"]=np.where(out.valid_votes.gt(0),out.total_main_blocs/out.valid_votes*100,np.nan)
    out["Other_raw_pct"]=np.where(out.valid_votes.gt(0),out.Other/out.valid_votes*100,np.nan)
    for b in MODELED_BLOCS:
        out[f"{b}_valid_pct"]=np.where(out.valid_votes.gt(0),out[b]/out.valid_votes*100,np.nan)
        out[f"{b}_pct"]=np.where(out.total_main_blocs.gt(0),out[b]/out.total_main_blocs*100,np.nan)
    return out

processed={e:aggregate_election(df,e) for e,df in raw.items()}
election_bloc_results=pd.concat(processed.values(),ignore_index=True).sort_values(["election_number","locality_symbol"]).reset_index(drop=True)
print(election_bloc_results.groupby("target_election").size())

## 5. Mapping and coverage audits

In [ ]:
lookup={(r.target_election,r.ballot_letter):(r.party_name,r.bloc) for r in party_mapping.itertuples()}
audit=[]
for election,df in raw.items():
    valid=num(df[first_col(df,["כשרים"])]).sum()
    for c in party_cols(df):
        votes=num(df[c]).fillna(0).sum(); info=lookup.get((election,c))
        audit.append({"target_election":election,"election_number":ELECTION_METADATA[election]["election_number"],"ballot_letter":c,"mapped":info is not None,"party_name":info[0] if info else pd.NA,"bloc":info[1] if info else "Other","votes":int(votes),"share_of_valid_votes_pct":votes/valid*100 if valid else np.nan})
mapping_audit=pd.DataFrame(audit).sort_values(["election_number","mapped","share_of_valid_votes_pct"],ascending=[True,True,False])
large_unmapped=mapping_audit.query("not mapped and share_of_valid_votes_pct > 2.0")
if not large_unmapped.empty: raise ValueError("Unmapped list above 2%:\n"+large_unmapped.to_string(index=False))
coverage=(election_bloc_results.groupby(["target_election","election_number","year"],as_index=False).agg(locality_rows=("locality_symbol","size"),unique_localities=("locality_symbol","nunique"),eligible_voters=("eligible_voters","sum"),votes_cast=("votes_cast","sum"),valid_votes=("valid_votes","sum"),modeled_votes=("total_main_blocs","sum"),other_votes=("Other","sum")))
coverage["modeled_vote_coverage_pct"]=coverage.modeled_votes/coverage.valid_votes*100
coverage["national_turnout_pct"]=coverage.votes_cast/coverage.eligible_voters*100
coverage.sort_values("election_number")

## 6. Save outputs

In [ ]:
paths={
"election":INTERIM_DIR/"election_bloc_results.csv",
"mapping":TABLES_DIR/"party_mapping_long.csv",
"mapping_audit":TABLES_DIR/"election_mapping_audit.csv",
"coverage":TABLES_DIR/"election_coverage_summary.csv",
"crosswalk":TABLES_DIR/"k17_locality_crosswalk_audit.csv",
"inventory":TABLES_DIR/"raw_source_inventory.csv",
"summary":SUMMARIES_DIR/"notebook_01_summary.json"}
election_bloc_results.to_csv(paths["election"],index=False,encoding="utf-8-sig")
party_mapping.to_csv(paths["mapping"],index=False,encoding="utf-8-sig")
mapping_audit.to_csv(paths["mapping_audit"],index=False,encoding="utf-8-sig")
coverage.to_csv(paths["coverage"],index=False,encoding="utf-8-sig")
pd.DataFrame(inventory).to_csv(paths["inventory"],index=False,encoding="utf-8-sig")
k17=raw["Knesset_17"]; s,n,m=attach_symbol(k17,"Knesset_17")
pd.DataFrame({"original_name":n,"normalized_name":n.map(norm_name),"locality_symbol":s,"match_method":m}).drop_duplicates().to_csv(paths["crosswalk"],index=False,encoding="utf-8-sig")
summary={"notebook":"01_data_loading_and_party_mapping","elections_processed":10,"rows_created":int(len(election_bloc_results)),"unique_localities":int(election_bloc_results.locality_symbol.nunique()),"largest_unmapped_list_pct":float(mapping_audit.loc[~mapping_audit.mapped,"share_of_valid_votes_pct"].max()),"outputs":{k:str(v.relative_to(REPO_ROOT)) for k,v in paths.items() if k!="summary"}}
paths["summary"].write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print("Saved",len(paths),"outputs")

### Completion check
Review the mapping audit, especially old elections. Do not continue if a major list is left in `Other` or if Knesset 17 has unresolved locality symbols.